### Import

In [2]:
import pandas as pd
import mlflow

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

In [3]:
df = pd.read_csv("../data/processed/rossmannV2.csv")

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("rossmann-forecasting")

2026/08/25 12:06:39 INFO mlflow.tracking.fluent: Experiment with name 'rossmann-forecasting' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1787652399876, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787652399876, lifecycle_stage='active', name='rossmann-forecasting', tags={}, trace_location=None, workspace='default'>

---

### Splitting the data

In [4]:
df = df.sort_values("Date").reset_index(drop=True)

cutoff = "2015-01-01"

train = df[df["Date"] < cutoff].copy()
test = df[df["Date"] >= cutoff].copy()

In [ ]:
X_train = train.drop(columns=["Sales", "Date"])
y_train = train["Sales"]

X_test = test.drop(columns=["Sales", "Date"])
y_test = test["Sales"]

In [6]:
tscv = TimeSeriesSplit(
    n_splits=5,
    gap=7
)

Here i added gap for so i have 7 rows gap between splits but it'll split wrong several times since dataset has several stores.

In [7]:
for train_idx, valid_idx in tscv.split(X_train):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_valid = X_train.iloc[valid_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_valid = y_train.iloc[valid_idx]

    print(
        X_fold_train["Date"].min(),
        X_fold_train["Date"].max(),
        "→",
        X_fold_valid["Date"].min(),
        X_fold_valid["Date"].max(),
    )

2013-01-08 2013-05-03 → 2013-05-03 2013-08-27
2013-01-08 2013-08-27 → 2013-08-27 2013-12-20
2013-01-08 2013-12-20 → 2013-12-20 2014-04-15
2013-01-08 2014-04-15 → 2014-04-15 2014-08-16
2013-01-08 2014-08-16 → 2014-08-16 2014-12-31


Prevents future data leaking.

---

### Baseline and Model choosing

In [10]:
ridge_params = {
    'alpha': 1.0
}

In [11]:
ridge_pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("model", Ridge(**ridge_params))
])